In [1]:
import requests
import pandas as pd
from pathlib import Path

In [2]:
import sys
sys.path.append("..")

from API_Key import gnews_API

In [3]:
# added a few more queries than before so we actually pull enough volume
queries = [
    "SAP",
    "SAP AI",
    "SAP ERP",
    "SAP cloud",
    '"SAP S/4HANA"',
    "SAP earnings",
    "SAP partnership",
    "enterprise software",
    "business AI",
    "SAP Technologies",
    "SAP competitors",
    "ERP market"
]

In [4]:
API_KEY = gnews_API

In [5]:
import time

In [6]:
all_articles = []

for q in queries:

    url = (
        f"https://gnews.io/api/v4/search?"
        f"q={q}&lang=en&max=10&apikey={API_KEY}"
    )

    response = requests.get(url)
    time.sleep(2)
    data = response.json()

    if "articles" in data:
        all_articles.extend(data["articles"])
    else:
        print(f"Query failed: {q}")
        print(data)

In [7]:
len(all_articles)

54

In [8]:
gnews_docs = []

for article in all_articles:

    gnews_docs.append({
        "title": article["title"],
        "description": article["description"],
        "content": article["content"],
        "source": article["source"]["name"],
        "published": article["publishedAt"],
        "url": article["url"]
    })


In [9]:
gnews_df = pd.DataFrame(gnews_docs)

len(gnews_df)

54

Removing Duplicates

In [10]:
print("Before:", len(gnews_df))

gnews_df = gnews_df.drop_duplicates(
    subset=["title"]
)

print("After:", len(gnews_df))

Before: 54
After: 46


In [11]:
gnews_df.head(1)

,title,description,content,source,published,url
0,INFRONEER Teams with Accenture to Build New Fi...,22.06.2026 - INFRONEER Holdings Inc. ('INFRONE...,"INFRONEER Holdings Inc. (""INFRONEER HD""), in c...",Wallstreet Online,2026-06-22T02:00:07Z,https://www.wallstreet-online.de/nachricht/210...


In [12]:
from rapidfuzz import fuzz

titles = gnews_df["title"].fillna("").tolist()

to_drop = set()

print("Before:", len(gnews_df))
for i in range(len(titles)):
    if i in to_drop:
        continue

    for j in range(i + 1, len(titles)):
        score = fuzz.ratio(titles[i], titles[j])

        if score >= 80:
            to_drop.add(j)
try:
    gnews_df = gnews_df.drop(index=list(to_drop)).reset_index(drop=True)
except KeyError as e:
    print("KeyError:", e)
    print("DataFrame indices:", gnews_df.index.tolist())
    print("Indices to drop:", sorted(to_drop))
print("After :", len(gnews_df))

Before: 46
KeyError: '[14] not found in axis'
DataFrame indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 15, 16, 17, 18, 19, 20, 21, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 51, 52]
Indices to drop: [3, 13, 14]
After : 46


In [13]:
data_dir = Path("../notebook/data")
if not data_dir.exists():
    data_dir = Path("notebook/data")
data_dir.mkdir(parents=True, exist_ok=True)

gnews_df.to_json(data_dir / "gnews_articles.json", orient="records", indent=2)

print("saved", len(gnews_df), "gnews articles to", data_dir / "gnews_articles.json")

saved 46 gnews articles to ../notebook/data/gnews_articles.json
